# Robot obstacle course: can examples help Jev?
Generate 300 fictional robots, then compare prediction quality and token use. **Enter your key below, then Restart Kernel and Run All.** No dataset download is needed.

The simulator creates correlated measurements and fixed interactions, with a little outcome noise. It contains 150 `finished` and 150 `failed` outcomes. Fresh samples avoid exact-row memorization; familiar physical relationships may still let Jev predict without examples. The sanity check measures this.

Install once in your terminal: `pip install -e ".[notebooks]"`.
Set `RUN_EXPERIMENT=False` for a free preview of the data, local baselines and token estimates. Live mode runs every section in order; results are cached.

## 1. Configuration

In [ ]:
API_KEY = ""  # Paste your Jev key here. Do not commit a notebook containing it.
MODEL = "jev-1.13.0"
CONTEXT_LIMIT = 32_000
SEED = 42
RUN_EXPERIMENT = True
MAX_TRAINING_CALLS = 640  # Shared by tree growth and pruning; counts new provider calls.
TOKEN_MARGIN = 1.2
MAX_TOTAL_TOKENS = None
CALIBRATION_FRACTION = .5
PRUNING_FRACTION = .3

import sys, math, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "examples"))
from robot_course_data import make_robot_course, split_robot_course, check_baselines
from xjevboost import XJevBoostClassifier, PredictionCache, Task
from xjevboost.jev import JevProvider, JevContextLimitError
from xjevboost.providers import normalized
from xjevboost.views import View, ViewBuilder, table
from xjevboost.splitting import stratified_split
LOCAL = ROOT / ".jupyter-local"
LOCAL.mkdir(exist_ok=True)
if RUN_EXPERIMENT and not API_KEY.strip():
    raise ValueError("Enter API_KEY above, or set RUN_EXPERIMENT=False for the free preview.")
provider = JevProvider(model=MODEL, context_limit=CONTEXT_LIMIT, api_key=API_KEY) if API_KEY.strip() else None
estimator = provider or JevProvider(model=MODEL, context_limit=CONTEXT_LIMIT)
cache = PredictionCache(LOCAL / "robot-course-jev.sqlite")
print("Live experiment" if RUN_EXPERIMENT else "Preview: no API calls")

## 2. Data, feature meanings and splits
The 10 columns contain two categorical fields, two ordered categories, four measurements with units in their names, and two booleans. Maintenance runs `poor → fair → good`; difficulty runs `easy → moderate → hard`. The hidden simulator rule is never sent to Jev.

The 200 training rows are divided automatically: **40 labeled examples**, **100 growth queries** to learn the tree, and **60 pruning queries** to check its branches. The 20 sanity rows compare context choices. The final 80 test rows are reserved for the final comparison. These are distinct roles, not multiple test sets. Every role is stratified; the seed makes the splits repeatable.

In [ ]:
X, y = make_robot_course(n_samples=300, random_state=SEED)
splits = split_robot_course(X, y, random_state=SEED)
X_train, y_train = splits["train"]
X_sanity, y_sanity = splits["sanity"]
X_test, y_test = splits["test"]
classes = tuple(sorted(y_train.unique()))
class_index = {label: i for i, label in enumerate(classes)}
train_rows, columns, _ = table(X_train)
ids = np.arange(len(X_train))
pool, cal_ids = stratified_split(train_rows, y_train.to_numpy(), ids, math.ceil(len(ids)*CALIBRATION_FRACTION), SEED)
example_ids, prune_ids = stratified_split(train_rows, y_train.to_numpy(), pool, math.ceil(len(ids)*PRUNING_FRACTION), SEED)
X_examples, y_examples = X_train.iloc[example_ids], y_train.iloc[example_ids]
roles = {"Examples": y_examples, "Growth": y_train.iloc[cal_ids], "Pruning": y_train.iloc[prune_ids], "Sanity": y_sanity, "Final test": y_test}
display(pd.DataFrame({name: labels.value_counts() for name, labels in roles.items()}).T)
display(X_examples.head(8))
display(pd.DataFrame({"dtype": X.dtypes.astype(str)}))
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
for label in classes:
    mask = y_train == label
    axes[0].scatter(X_train.loc[mask, "motor_torque_nm"], X_train.loc[mask, "payload_weight_kg"], label=label, alpha=.6)
axes[0].set(xlabel="Motor torque (Nm)", ylabel="Payload weight (kg)", title="Correlated training measurements")
axes[0].legend()
pd.crosstab(X_train.course_surface, y_train, normalize="index").plot.bar(ax=axes[1])
axes[1].set(title="Training outcomes by surface", ylabel="Fraction of robots")
plt.show()
print("Five-fold cross-validation on training rows only:")
display(pd.DataFrame(check_baselines(X_train, y_train)).T)

## 3. Task and token preview
Full context means all **40 example rows × 10 columns**, plus one query. The tree uses subsets of that pool. No provider calls happen in this section.

Estimates include instructions and output allowance. The 20% margin is conservative, but Jev's estimator is approximate: this is an **estimated ceiling**, not a guaranteed actual-token limit. The tree checks its cumulative budget before each call. Optional full-context checks are skipped with a warning if they cannot fit.

In [ ]:
instructions = (
    "Predict whether a robot finishes a fictional obstacle course. "
    "Use the supplied labeled examples to learn the relationships between measurements and outcomes. "
    "Maintenance condition is ordered poor, fair, good; obstacle difficulty is ordered easy, moderate, hard. "
    "Numeric units are in column names. Boolean fields indicate whether equipment is present/enabled. "
    "The simulation includes outcome noise; do not assume any single feature guarantees the outcome."
)
descriptions = {"failed": "The robot did not finish the course.", "finished": "The robot finished the course."}
task = Task(classes, instructions, tuple(descriptions[c] for c in classes))
example_rows, columns, _ = table(X_examples)
builder = ViewBuilder(example_rows, y_examples.map(class_index).to_numpy(), columns)
preview = []
for name, queries in [("Sanity", X_sanity), ("Growth", X_train.iloc[cal_ids]), ("Pruning", X_train.iloc[prune_ids]), ("Test", X_test)]:
    rows, _, _ = table(queries, columns)
    for mode in ["Full context", "No examples"]:
        views = [builder.full(r) if mode == "Full context" else View(columns, (), r) for r in rows]
        estimates = [estimator.estimate(v, task).total for v in views]
        preview.append({"role": name, "mode": mode, "rows": len(rows), "mean_estimated_tokens_per_call": np.mean(estimates),
                        "max_reserved_tokens": max(math.ceil(t*TOKEN_MARGIN) for t in estimates),
                        "estimated_total_tokens": sum(estimates)})
display(pd.DataFrame(preview))
print("Sanity: at most 40 calls. Tree fit: at most", MAX_TRAINING_CALLS, "new calls.")
print("Test: up to 160 adaptive calls plus 80 full and 80 zero-example calls; root-only evaluation reuses the cache.")

## 4. Sanity check: do labeled examples help?
All methods below see the same 20 sanity queries. At this size, one correct answer changes accuracy by 5 percentage points. A balanced majority baseline gets 50%. Compare Brier loss too: **lower is better** and it penalizes confidently wrong predictions.

Precision measures how often a predicted success is correct; recall measures how many actual successes were found. Failure recall checks the other class. This notebook continues through the experiment automatically; the sanity result is evidence to interpret, not a guarantee that context will help.

In [ ]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, log_loss

def metrics(probabilities, truth):
    truth = np.asarray(truth)
    prediction = np.asarray(classes)[np.argmax(probabilities, axis=1)]
    targets = np.eye(len(classes))[[class_index[v] for v in truth]]
    return {"accuracy": accuracy_score(truth, prediction), "balanced_accuracy": balanced_accuracy_score(truth, prediction),
            "brier": float(np.mean((probabilities-targets)**2)), "log_loss": log_loss(truth, probabilities, labels=list(classes)),
            "success_precision": precision_score(truth, prediction, pos_label="finished", zero_division=0),
            "success_recall": recall_score(truth, prediction, pos_label="finished", zero_division=0),
            "failure_recall": recall_score(truth, prediction, pos_label="failed", zero_division=0),
            "macro_f1": f1_score(truth, prediction, average="macro", zero_division=0)}

prior = y_examples.value_counts().reindex(classes, fill_value=0).to_numpy()/len(y_examples)
def context_check(queries, truth, use_examples, name):
    rows, _, _ = table(queries, columns)
    views = [builder.full(r) if use_examples else View(columns, (), r) for r in rows]
    if any(math.ceil(provider.estimate(v, task).total*TOKEN_MARGIN) > CONTEXT_LIMIT for v in views):
        warnings.warn(name + " skipped: estimated context is too large.")
        return None
    predictions, usage, estimates, hits = [], [], [], 0
    for i, view in enumerate(views):
        key = cache.key(provider, task, view)
        result = cache.get(key)
        if result is None:
            try:
                result = normalized(provider.predict(view, task), len(classes))
            except JevContextLimitError:
                warnings.warn(name + " skipped: provider rejected context size; no partial score reported.")
                return None
            cache.put(key, result)
        else:
            hits += 1
        predictions.append(result.probabilities)
        usage.append(result.actual_tokens)
        estimates.append(provider.estimate(view, task).total)
        print(f"{name}: {i+1}/{len(views)}", flush=True)
    result = metrics(np.asarray(predictions), truth)
    result.update(average_calls=1, average_actual_tokens_per_query=np.mean(usage) if all(v is not None for v in usage) else None,
                  average_estimated_tokens_per_query=np.mean(estimates), new_calls=len(rows)-hits, cache_hits=hits)
    return result

sanity_reports = {"Example prior": metrics(np.tile(prior, (len(y_sanity), 1)), y_sanity)}
if RUN_EXPERIMENT:
    for name, examples in [("No examples", False), ("Full context", True)]:
        report = context_check(X_sanity, y_sanity, examples, "Sanity " + name)
        if report is not None: sanity_reports[name] = report
display(pd.DataFrame(sanity_reports).T)

## 5. Real run: learn the adaptive tree
Four candidate views, at most two calls per query. Each sampled view contains every class. Growth/pruning queries never appear among the labeled examples. Progress logs show completed calls, not a misleading percentage of a call budget.

`calibration_fraction=.5` and `pruning_fraction=.3` refer to the original 200 training rows. The remaining 20% supplies the fixed examples.

Updated search: ordinary views use **8 columns and 10 labeled examples**, emphasizing feature coverage over example count. Previous training and test outputs were cleared because they used different settings. Sanity checks still use the unchanged full example pool.

`query_columns="mixed"` chooses matching or all query columns independently with 50% probability per candidate recipe. Example columns stay sampled. The seeded choice is fixed for growth, pruning and prediction; four candidates need not include both modes.

**Shared training budget:** `MAX_TRAINING_CALLS` covers both growth and pruning. Growth can leave too few calls for pruning; branches that cannot be validated may then be removed, potentially leaving only the root view. The cap is 640 here, covering the maximum 4 views × (100 growth + 60 pruning rows). Sanity and test calls are additional; cached responses do not consume new calls.

In [ ]:
clf = None
if RUN_EXPERIMENT:
    clf = XJevBoostClassifier(provider=provider, task_instructions=instructions, class_descriptions=descriptions,
        calibration_fraction=CALIBRATION_FRACTION, pruning_fraction=PRUNING_FRACTION,
        n_views=4, subsample=.25, colsample_bytree=.8, query_columns="mixed", max_depth=2,
        min_samples_leaf=2, screening_samples=8, random_state=SEED,
        max_training_calls=MAX_TRAINING_CALLS, cache=cache,
        full_view_budget=True, token_margin=TOKEN_MARGIN, max_total_tokens=MAX_TOTAL_TOKENS)
    clf.fit(X_train, y_train, verbose=True)
    assert np.array_equal(clf.example_pool_indices_, example_ids)
    assert np.array_equal(clf.calibration_indices_, cal_ids)
    assert np.array_equal(clf.pruning_indices_, prune_ids)
else:
    print("Preview mode: live training skipped.")

## 6. Training costs and tree contents
Training cost includes candidate screening, even for views the final tree does not use. Savings against the dense query–view grid describe training work avoided, not inference savings.

Leaf path costs describe the finished policy on growth rows and include every acquisition along that path. Growth scores are training scores; pruning scores influenced policy selection. Neither is unbiased test performance.

In [ ]:
if clf is not None:
    display(pd.Series(clf.training_stats_, name="Training statistics"))
    display(pd.DataFrame(clf.training_leaf_stats_))
    clf.plot_tree()
    plt.show()

## 7. Final evaluation on the same 80 held-out rows
**Single view** requests only the fitted root. **Adaptive** may acquire a second view depending on its first answer. **Full context** requests all 40 examples and all columns. **No examples** measures Jev's predictions without labeled context.

Random Forest and Extra Trees use all 200 training rows, with categorical values one-hot encoded inside their pipelines. They are conventional ML references, not equal-context Jev comparisons. All settings are fixed before evaluating this test set. Avoid tuning to its results.

In [ ]:
reports = {}
inference_reports = {}
if RUN_EXPERIMENT:
    from sklearn.compose import ColumnTransformer
    from sklearn.preprocessing import OneHotEncoder
    from sklearn.pipeline import make_pipeline
    from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
    reports["Example prior"] = metrics(np.tile(prior, (len(y_test), 1)), y_test)
    for name, model in [("Random forest", RandomForestClassifier(n_estimators=300, min_samples_leaf=2, random_state=SEED, n_jobs=-1)),
                        ("Extra trees", ExtraTreesClassifier(n_estimators=300, min_samples_leaf=2, random_state=SEED, n_jobs=-1))]:
        pipeline = make_pipeline(ColumnTransformer([("categories", OneHotEncoder(handle_unknown="ignore"), X_train.select_dtypes("category").columns.tolist())], remainder="passthrough"), model)
        pipeline.fit(X_train, y_train)
        assert tuple(pipeline.classes_) == classes
        reports[name] = metrics(pipeline.predict_proba(X_test), y_test)
    for name, policy in [("Adaptive", "adaptive"), ("Single view", "single_view")]:
        summary = clf.evaluate(X_test, y_test, policy=policy)
        inference_reports[name] = summary
        success = list(clf.classes_).index("finished")
        failure = list(clf.classes_).index("failed")
        reports[name] = {k: summary[k] for k in ["accuracy", "brier", "log_loss"]}
        reports[name].update(balanced_accuracy=float(np.mean(summary["recall_per_class"])),
            success_precision=summary["precision_per_class"][success], success_recall=summary["recall_per_class"][success],
            failure_recall=summary["recall_per_class"][failure], macro_f1=float(np.mean(summary["f1_per_class"])))
        for key in ["average_calls", "average_actual_tokens_per_query", "average_accounted_tokens_per_query", "token_reduction_vs_full_estimate_pct"]:
            reports[name][key] = summary[key]
    for name, examples in [("Full context", True), ("No examples", False)]:
        result = context_check(X_test, y_test, examples, "Test " + name)
        if result is not None: reports[name] = result
    display(pd.DataFrame(reports).T)
else:
    print("Preview mode: final test evaluation skipped.")

## 8. Reading the tradeoff and saving the policy
The goal is to approach full-context quality using fewer tokens. Accuracy is higher-is-better; Brier and log loss are lower-is-better. A positive adaptive-minus-full Brier gap means adaptive probabilities are worse.

Actual tokens come from provider usage; unavailable usage stays missing. Accounted tokens may include estimates. Savings against a hypothetical full request are different from savings against a measured full benchmark. Cached calls still count toward the policy's logical cost, even though rerunning does not spend those tokens again.

Saved JSON includes the fixed labeled examples and tree, but excludes provider credentials. Pass a provider when loading.

In [ ]:
if clf is not None:
    for name, summary in inference_reports.items():
        print(name, "call-count distribution:", summary["call_count_distribution"])
        display(pd.DataFrame(summary["leaf_stats"]))
    if "Full context" in reports:
        print("Adaptive minus full Brier:", reports["Adaptive"]["brier"] - reports["Full context"]["brier"])
        full_tokens = reports["Full context"].get("average_actual_tokens_per_query")
        adaptive_tokens = reports["Adaptive"].get("average_actual_tokens_per_query")
        if full_tokens and adaptive_tokens is not None:
            print(f"Measured token saving versus full context: {100*(1-adaptive_tokens/full_tokens):.1f}%")
    path = LOCAL / "robot-course-model.json"
    clf.save_model(path)
    loaded = XJevBoostClassifier.load_model(path, provider=provider, cache=cache)
    print("Saved model:", path)
    display(pd.DataFrame(clf.predict_with_trace(X_test.iloc[:3])))

## 9. Savings in context: what did we give up?
This table uses **reported actual tokens**, not the hypothetical full-request estimate. Positive savings mean cheaper; negative savings mean more expensive. Accuracy differences are **percentage points** (positive is better); Brier differences are raw loss differences (negative is better). Unknown usage stays missing.

Compare against both full context and no examples: a policy can save tokens versus full context while costing more and predicting worse than zero-example Jev. Calls count logical acquisitions, including cached results, so the comparison does not change merely because a notebook is rerun. Forests and the prior use no Jev calls; they are omitted from this API-token table because their costs are local computation.

`colsample_bytree=0.8` means ordinary candidates receive 8/10 example columns, while the broader candidate receives all 10. `subsample=0.25` means ordinary candidates receive 10/40 labeled examples (the broader candidate uses 15/40). Full context uses 40 examples and 10 columns; no examples uses zero examples and 10 columns. These fractions are **not token fractions**.

Training expenditure is listed separately. Token break-even means how many future queries would recover this run's new training tokens through inference savings alone. It excludes sanity/benchmark calls, ignores prediction-quality differences, and is not a claim that a less accurate policy is worthwhile. Cached refits may have zero new training expenditure.

With `query_columns="mixed"`, each recipe sends either its example-column subset or all 10 query columns. Inspect the tree and trace `query_columns` to see the actual coverage; this is separate from the example-column ratio.

In [ ]:
def savings_in_context(reports, references=("Full context", "No examples")):
    rows = []
    for name in ("Full context", "No examples", "Single view", "Adaptive"):
        if name not in reports:
            continue
        result = reports[name]
        tokens = result.get("average_actual_tokens_per_query")
        row = {"method": name, "accuracy (%)": 100 * result["accuracy"], "Brier": result["brier"],
               "calls/query": result.get("average_calls"), "actual tokens/query": tokens}
        for reference in references:
            baseline = reports.get(reference)
            if baseline is None:
                continue
            base_tokens = baseline.get("average_actual_tokens_per_query")
            row[f"tokens saved vs {reference} (%)"] = (
                100 * (1 - tokens / base_tokens)
                if tokens is not None and pd.notna(tokens) and base_tokens is not None and pd.notna(base_tokens) and base_tokens > 0 else None)
            row[f"accuracy vs {reference} (pp)"] = 100 * (result["accuracy"] - baseline["accuracy"])
            row[f"Brier vs {reference}"] = result["brier"] - baseline["brier"]
        rows.append(row)
    return pd.DataFrame(rows).set_index("method") if rows else pd.DataFrame()

if reports:
    display(savings_in_context(reports).round(3))
else:
    print("Preview mode: no live evaluation results to compare.")

if clf is not None:
    s = clf.training_stats_
    display(pd.Series({
        "New training provider calls": s["provider_calls"],
        "Actual training tokens (new requests)": s["actual_tokens"] if s["calls_without_usage"] == 0 else None,
        "Dense training query-view grid (possible pairs)": s["query_view_upper_bound"],
        "Training query-view pairs evaluated": s["unique_query_view_pairs"],
        "Training grid not evaluated (%)": s["screening_reduction_pct"],
        "Training call budget exhausted": s["budget_exhausted"],
    }, name="Training expenditure"))
    training_tokens = s["actual_tokens"] if s["calls_without_usage"] == 0 else None
    adaptive_tokens = reports.get("Adaptive", {}).get("average_actual_tokens_per_query")
    for reference in ("Full context", "No examples"):
        baseline_tokens = reports.get(reference, {}).get("average_actual_tokens_per_query")
        if any(v is None or pd.isna(v) for v in (training_tokens, adaptive_tokens, baseline_tokens)):
            print(f"Token break-even vs {reference}: unavailable (missing actual usage).")
        elif baseline_tokens <= adaptive_tokens:
            print(f"Token break-even vs {reference}: none; adaptive does not save inference tokens.")
        else:
            queries = math.ceil(training_tokens / (baseline_tokens - adaptive_tokens))
            print(f"Token-only break-even vs {reference}: {queries:,} future queries (quality differences still apply).")
